In [1]:
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch import Tensor
import matplotlib.pyplot as plt
from torchvision import datasets
from torchvision import transforms
from torch.utils.data import random_split

In [2]:
# 1. 사용할 디바이스 지정 (NVIDIA GPU -> Mac GPU -> CPU 순서)
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():  # M1/M2/M3 맥북 유저용
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"현재 사용 중인 디바이스: {device}")

현재 사용 중인 디바이스: cuda


### 학습 데이터 설정


In [3]:
train_dataset = datasets.MNIST(
    root="data/mnist_data/", train=True, transform=transforms.ToTensor(), download=True
)
test_dataset = datasets.MNIST(
    root="data/mnist_data/", train=False, transform=transforms.ToTensor(), download=True
)

In [4]:
train_dataset_size = int(len(train_dataset) * 0.80)
validation_dataset_size = len(train_dataset) - train_dataset_size

train_dataset, validation_dataset = random_split(
    train_dataset, [train_dataset_size, validation_dataset_size]
)

print(len(train_dataset), len(validation_dataset), len(test_dataset))

48000 12000 10000


In [5]:
BATCH_SIZE = 64

train_dataset_loader = DataLoader(
    dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True
)
validation_dataset_loader = DataLoader(
    dataset=validation_dataset, batch_size=BATCH_SIZE, shuffle=True
)
test_dataset_loader = DataLoader(
    dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False
)

### 모델 생성


In [6]:
class MlpModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.layer_stack = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 10),
        )

    def forward(self, data):
        logits = self.layer_stack(data)
        return logits

In [7]:
model = MlpModel()

loss_function = (
    nn.CrossEntropyLoss()
)  # CrossEntropyLoss 손실함수에는 softmax 함수 포함되어 있음
optim = torch.optim.SGD(model.parameters(), lr=1e-2)

In [8]:
from torchinfo import summary

summary(model, input_size=(1, 1, 28, 28))

Layer (type:depth-idx)                   Output Shape              Param #
MlpModel                                 [1, 10]                   --
├─Sequential: 1-1                        [1, 10]                   --
│    └─Flatten: 2-1                      [1, 784]                  --
│    └─Linear: 2-2                       [1, 256]                  200,960
│    └─ReLU: 2-3                         [1, 256]                  --
│    └─Dropout: 2-4                      [1, 256]                  --
│    └─Linear: 2-5                       [1, 10]                   2,570
Total params: 203,530
Trainable params: 203,530
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.20
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.81
Estimated Total Size (MB): 0.82

### 학습 및 검증 함수 선언


In [9]:
def model_train(
    dataloader: DataLoader,
    model: MlpModel,
    loss_function: nn.CrossEntropyLoss,
    optim: torch.optim.SGD,
):

    model.train()

    loss_sum = train_correct = total_count = 0
    batch_count = len(dataloader)

    for images, labels in dataloader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = loss_function(outputs, labels)

        optim.zero_grad()
        loss.backward()
        optim.step()

        loss_sum += loss.item()

        total_count += labels.size(0)
        train_correct += ((torch.argmax(outputs, 1) == labels)).sum().item()

    avg_loss = loss_sum / batch_count
    avg_accuracy = 100 * train_correct / total_count

    return (avg_loss, avg_accuracy)

In [10]:
def model_evaluate(
    dataloader: DataLoader,
    model: MlpModel,
    loss_function: nn.CrossEntropyLoss,
):

    model.eval()

    with torch.no_grad():

        loss_sum = correct_list = total_count = 0
        batch_count = len(dataloader)

        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = loss_function(outputs, labels)

            loss_sum += loss.item()

            total_count += labels.size(0)
            correct_list += ((torch.argmax(outputs, 1) == labels)).sum().item()

        avg_loss = loss_sum / batch_count
        avg_accuracy = 100 * correct_list / total_count

    return (avg_loss, avg_accuracy)

In [11]:
def model_test(
    dataloader: DataLoader,
    model: MlpModel,
    loss_function: nn.CrossEntropyLoss,
):

    model.eval()

    with torch.no_grad():

        loss_sum = correct_list = total_count = 0
        batch_count = len(dataloader)

        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = loss_function(outputs, labels)

            loss_sum += loss.item()

            total_count += labels.size(0)
            correct_list += ((torch.argmax(outputs, 1) == labels)).sum().item()

        avg_loss = loss_sum / batch_count
        avg_accuracy = 100 * correct_list / total_count

    print("accuracy:", avg_accuracy)
    print("loss:", avg_loss)

In [12]:
train_loss_list = []
train_accuracy_list = []

val_loss_list = []
val_accuracy_list = []

EPOCHS = 20

for epoch in range(EPOCHS):

    # ================== model train ======================
    train_avg_loss, train_avg_accuracy = model_train(
        train_dataset_loader, model, loss_function, optim
    )

    train_loss_list.append(train_avg_loss)
    train_accuracy_list.append(train_avg_accuracy)
    # =====================================================

    # ================== model evaluation =================
    val_avg_loss, val_avg_accuracy = model_evaluate(
        validation_dataset_loader, model, loss_function
    )

    val_loss_list.append(val_avg_loss)
    val_accuracy_list.append(val_avg_accuracy)
    # =====================================================

    print(
        f"epoch: {epoch}, train loss= {train_avg_loss}, train accuracy= {train_avg_accuracy},"
        f"validation loss= {val_avg_loss}, validation accuracy= {val_avg_accuracy}"
    )

epoch: 0, train loss= 1.4301007955869038, train accuracy= 68.56666666666666,validation loss= 0.7280188624529128, validation accuracy= 83.375
epoch: 1, train loss= 0.6058556067148845, train accuracy= 84.25416666666666,validation loss= 0.4806728738736599, validation accuracy= 87.56666666666666
epoch: 2, train loss= 0.4685539188782374, train accuracy= 87.03333333333333,validation loss= 0.4080631763060042, validation accuracy= 89.0
epoch: 3, train loss= 0.41530030641953153, train accuracy= 88.21666666666667,validation loss= 0.37099095045569097, validation accuracy= 89.625
epoch: 4, train loss= 0.3835471393068631, train accuracy= 89.05833333333334,validation loss= 0.347260325592249, validation accuracy= 90.14166666666667
epoch: 5, train loss= 0.358815833846728, train accuracy= 89.81041666666667,validation loss= 0.32887199647883153, validation accuracy= 90.75833333333334
epoch: 6, train loss= 0.33799290161331497, train accuracy= 90.39791666666666,validation loss= 0.313187662512064, validatio

### 모델 테스트

In [13]:
model_test(test_dataset_loader, model, loss_function)

accuracy: 94.82
loss: 0.1849020815092572
